# Tibeb AI — Fine-Tuning on Google Colab

Fine-tunes **CohereForAI/aya-expanse-8b** with QLoRA for Amharic financial AI.

**Requirements:** GPU runtime (T4 free tier works, A100 recommended)

**Runtime → Change runtime type → GPU (T4 or A100)**

## 1. Check GPU and Install Dependencies

In [ ]:
!nvidia-smi
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q transformers>=4.40.0 datasets>=2.18.0 peft>=0.10.0 trl>=0.8.0 \
    accelerate>=0.28.0 bitsandbytes>=0.43.0 huggingface_hub>=0.22.0 \
    safetensors>=0.4.0 wandb

## 2. Upload Training Data

Upload `tibeb_unified_train.jsonl` from your local machine.

**Option A:** Use the file upload button below (slow for large files)

**Option B:** Upload to Google Drive first, then mount

In [ ]:
# Option A: Direct upload (works for files up to ~1.5GB)
# from google.colab import files
# uploaded = files.upload()  # Select tibeb_unified_train.jsonl

# Option B: Google Drive (recommended for 1.65GB file)
from google.colab import drive
drive.mount('/content/drive')

# Copy from Drive (adjust path to where you uploaded the file)
!cp "/content/drive/MyDrive/tibeb_unified_train.jsonl" /content/tibeb_unified_train.jsonl
!wc -l /content/tibeb_unified_train.jsonl

## 3. Load and Prepare Data

In [ ]:
import json

DATASET_PATH = "/content/tibeb_unified_train.jsonl"
MODEL_ID = "CohereForAI/aya-expanse-8b"

def format_row(row):
    instruction = row.get("instruction", "")
    inp = row.get("input", "")
    output = row.get("output", "")
    user_msg = f"{instruction}\n\n{inp}" if inp else instruction
    return {"messages": [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": output},
    ]}

rows = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if not row.get("instruction") or not row.get("output"):
            continue
        # Skip very short outputs
        if len(row["output"].strip()) < 10:
            continue
        rows.append(format_row(row))

print(f"Loaded {len(rows):,} training rows")

# Train/val split
split_idx = int(len(rows) * 0.95)
train_rows = rows[:split_idx]
eval_rows = rows[split_idx:]
print(f"Train: {len(train_rows):,}, Val: {len(eval_rows):,}")

## 4. Load Model (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Configure Training

Settings auto-adapt to your GPU:
- **T4 (16GB):** batch=1, grad_accum=16, ~2-3 it/sec
- **A100 (40GB):** batch=4, grad_accum=4, ~8-12 it/sec

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import Dataset

# Auto-detect GPU and set config
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0)

if vram_gb >= 35:  # A100
    batch_size = 4
    grad_accum = 4
    max_seq = 1024
    print(f"A100 config: batch={batch_size}, seq={max_seq}")
elif vram_gb >= 20:  # A10G / L4
    batch_size = 2
    grad_accum = 8
    max_seq = 768
    print(f"A10G/L4 config: batch={batch_size}, seq={max_seq}")
else:  # T4
    batch_size = 1
    grad_accum = 16
    max_seq = 512
    print(f"T4 config: batch={batch_size}, seq={max_seq}")

OUTPUT_DIR = "/content/tibeb-sft-adapter"

def format_chat(example):
    return tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )

train_ds = Dataset.from_list(train_rows)
eval_ds = Dataset.from_list(eval_rows)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_total_limit=3,
    max_grad_norm=1.0,
    report_to="none",
    seed=42,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

print(f"\nGPU: {gpu_name} ({vram_gb:.0f}GB)")
print(f"Train: {len(train_ds):,} rows")
print(f"Effective batch: {batch_size * grad_accum}")
print(f"Max seq length: {max_seq}")

## 6. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=format_chat,
    max_seq_length=max_seq,
    packing=True,
)

print("Starting training...\n")
trainer.train()

# Save final adapter
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nAdapter saved to: {OUTPUT_DIR}")

## 7. Test the Model

In [ ]:
from peft import PeftModel

# Reload for inference
del model
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(model, OUTPUT_DIR)

prompts = [
    "ሰላም! ቲ-ቢል ምንድን ነው?",
    "አክሲዮን ገበያ ላይ ኢንቨስት ማድረግ እፈልጋለሁ። ከየት ልጀምር?",
    "የምንዛሬ ተመን ለቁጠባዬ ምን ተጽዕኖ አለው?",
    "ለጀማሪ ኢንቨስተር ምን ትመክራለህ?",
]

for p in prompts:
    messages = [{"role": "user", "content": p}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"USER: {p}")
    print(f"TIBEB: {response}")
    print("─" * 60)

## 8. Save Adapter

Download the trained adapter to your local machine, or push to HuggingFace.

In [ ]:
# Option A: Download to local machine
import shutil
shutil.make_archive("/content/tibeb-sft-adapter", "zip", OUTPUT_DIR)
from google.colab import files
files.download("/content/tibeb-sft-adapter.zip")

In [ ]:
# Option B: Copy to Google Drive
!cp -r /content/tibeb-sft-adapter /content/drive/MyDrive/tibeb-sft-adapter
print("Saved to Google Drive: MyDrive/tibeb-sft-adapter")

In [ ]:
# Option C: Push to HuggingFace
from huggingface_hub import HfApi, login

# login(token="your_hf_token")  # or use notebook_login()
from huggingface_hub import notebook_login
notebook_login()

REPO_ID = "nahommohan/tibeb-aya-8b"  

api = HfApi()
api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"Pushed to: https://huggingface.co/{REPO_ID}")